In [1]:
import numpy as np
from pyscf import gto, scf, cc
import jax
jax.config.update("jax_enable_x64", True)

d = 100

natom = 2
atoms = ""
for n in range(natom):
    shift = n*d
    atoms += f'N {0.0+shift} 0.0 0.0 \n'
    atoms += f'N {0.0+shift} 0.0 2.4 \n'

spin = 0
mol = gto.M(atom=atoms, 
            basis="sto6g", 
            spin=spin, 
            unit='B',
            verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()
    
stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'UHF Energy: {mf.e_tot}, stability {stable}')
        break


mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()

print(mycc.energy(mycc.t1, mycc.t2*0))

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#8737-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Tue Aug  4 21:22:27 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 4
[INPUT] num. electrons = 28
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X  

In [2]:
import time
import numpy as np
from jax import numpy as jnp
from jax import jit
import opt_einsum as oe

from afqmc import config
from afqmc import prep

from functools import partial
print = partial(print, flush=True)
config.setup_jax()

Hostname:     yichi-thinkpad
System:       Linux
Node:         yichi-thinkpad
Release:      4.4.0-26100-Microsoft
Machine:      x86_64
Processor:    x86_64
JAX backend:  CPU
JAX devices:  [CpuDevice(id=0)]
Device kind:  cpu
Platform:     cpu


In [4]:
from afqmc import integral
integral.prep_integral(mycc, chol_cut=1e-8)


Preparing AFQMC calculation
CCSD type input object
Calculating Cholesky integrals
Cholesky shape: (102, 16, 16) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [10, 10]
Number of basis functions:  16
Number of Cholesky vectors: 102


In [5]:
options = {'eql_time': 10,
           'n_blocks': 100,
           'n_walkers': 10,
           'mix_precision': False,
           'seed': 17,
           'guide': 'rhf',
           'trial': 'rpt2ccsd_bar',
           }

In [6]:
ham_data, ham, prop, trial, wave_data, sampler, options = prep.init_afqmc(options=options)
wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
prop_data = prep.init_hf_prop_data(trial, wave_data, ham_data, options)
print(mf.e_tot - prop_data["e_estimate"])


QMC Parameters
eql_time        -         10
n_blocks        -        100
n_walkers       -         10
mix_precision   -      False
seed            -         17
guide           -        rhf
trial           - rpt2ccsd_bar
dt              -      0.005
n_exp_terms     -          6
n_prop_steps    -         50
walker_type     -        rhf
n_batch         -          1
max_error       -          0
nchol_chunk     -        100
max_memory      -       2000
free_projection -      False

Load system from Integral File
Maximum memory per walker:            200.00 MB
Maximum number of Cholesky per chunk: 51200
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         102
Number of padding Cholesky:           0

QMC System
Number of electrons: (10, 10)
Spin Multiplicity:   0
Number of orbitals:  16
Number of Chol:      102

Initalize QMC walkers by HF
-9.752341156854527e-10


In [7]:
def pt2_energy_formula(h0, t2, e0, e1):
    return h0 + e0 + e1 - t2*e0

In [8]:
norb = trial.norb
h0 = ham_data["h0"]
h1 = ham_data["h1"][0]
chol = ham_data["chol"].reshape(-1, norb, norb)

In [9]:
walker_init = prop_data['walkers'][0]
obar, t2, e0, e1 = \
    trial._calc_energy_pt(walker_init, ham_data, wave_data)
print(pt2_energy_formula(h0, t2, e0, e1)-mycc.e_tot)


(9.947314083547099e-10+0j)


In [ ]:
from jax import jit, lax
import opt_einsum as oe

@partial(jit, static_argnums=0)
def _calc_energy_pt2_decomposed(self, walker, ham_data, wave_data):
    # becareful tau is complex!

    if self.mix_precision:
        rtype = jnp.float32
        ctype = jnp.complex64
    else:
        rtype = jnp.float64
        ctype = jnp.complex128
    
    nocc, norb = self.nelec[0], self.norb
    nchol_chunk = self.nchol_chunk  # nchol per chunk

    tau = wave_data["tau"]
    h1 = ham_data["h1_bar"]
    chol = ham_data["chol_bar"]
    walker_bar = wave_data['exp_t1'] @ walker

    obar = jnp.linalg.det(walker_bar[:walker_bar.shape[1], :]) ** 2

    green = (walker_bar.dot(jnp.linalg.inv(walker_bar[: walker_bar.shape[1], :]))).T
    green_occ = green[:, nocc:]
    greenp = jnp.vstack((green_occ, -jnp.eye(norb - nocc)))
    green_ov = green[:nocc,nocc:]

    rot_chol = chol[:, :nocc, :]
    nchol = chol.shape[0]
    # chunk_size = naux // nchol_chunk

    # 1 body energy
    hg = oe.contract("pi,pi->", h1[:nocc, :], green, backend="jax")
    e1_0 = 2 * hg

    taug = oe.contract("yia,ja->yij", tau, green_ov, backend="jax")
    taugp = oe.contract("yjb,pb->yjp", tau, greenp, backend="jax")
    taugpg = oe.contract("yjp,jq->ypq", taugp, green[:nocc,:], backend="jax")
    taugg = oe.contract("yji,jq->yiq", taug, green[:nocc,:], backend="jax")
    tr_taug = oe.contract("yii->y", taug, backend="jax")
    t2o_c = oe.contract("y,y->", tr_taug, tr_taug, backend="jax")
    t2o_e = oe.contract("yij,yji->", taug, taug, backend="jax")
    t2o = 2 * t2o_c - t2o_e # <HF|T2|walker>

    e1_2_1 = t2o * e1_0

    # t2g_c = oe.contract("iajb,ia->jb", t2, green[:nocc,nocc:], backend="jax")
    # t2g_e = oe.contract("iajb,ib->ja", t2, green[:nocc,nocc:], backend="jax")
    t2_green_c = oe.contract("y,ypq->pq", tr_taug, taugpg, backend="jax")
    t2_green_e = oe.contract("yip,yiq->pq", taugp, taugg, backend="jax")
    t2_green = 2 * t2_green_c - t2_green_e

    e1_2_2 = -2 * oe.contract("pq,pq->", h1, t2_green, backend="jax")
    e1_2 = e1_2_1 + e1_2_2 # <HF|T2 h1|walker>/<HF|walker>

    # pad with zero cholesky vectors — contributes nothing to any contraction
    npad = (-nchol) % nchol_chunk
    chol = jnp.concatenate([chol, jnp.zeros((npad, norb, norb))], axis=0)
    rot_chol = jnp.concatenate([rot_chol, jnp.zeros((npad, nocc, norb))], axis=0)

    # reshape into chunks: (n_chunks, chunk_size, ...)
    nchunk = (nchol + npad) // nchol_chunk
    chol = chol.reshape(nchunk, nchol_chunk, norb, norb)
    rot_chol = rot_chol.reshape(nchunk, nchol_chunk, nocc, norb)

    # two body — scan over chunks, explicit contractions within a chunk
    def scan_chunk(carry, x):
        chol_c, rot_chol_c = x  # (chunk_size, norb, norb), (chunk_size, nocc, norb)

        gl = oe.contract("ir,gqr->giq", green, chol_c, backend="jax")
        gl_c = oe.contract("gii->g", gl[:, :, :nocc], backend="jax")
        e2_0_c = oe.contract("g,g->", gl_c, gl_c, backend="jax") * 2
        e2_0_e = -oe.contract("gij,gji->", gl[:, :, :nocc], gl[:, :, :nocc], backend="jax")
        carry[0] += e2_0_c + e2_0_e

        lt2g = oe.contract("gpr,pr->g", 
                            chol_c.astype(rtype), 
                            t2_green.astype(ctype), 
                            backend="jax")
        carry[1] += -oe.contract("g,g->", 
                                    lt2g.astype(ctype), 
                                    gl_c.astype(ctype), 
                                    backend="jax")

        lt2_green = oe.contract("gir,qr->giq", 
                                rot_chol_c.astype(rtype), 
                                t2_green.astype(ctype), 
                                backend="jax")
        # t_iajb |G_ia G_js Gp_pb| G_qr L_pr L_qs
        carry[2] += 0.5 * oe.contract("giq,giq->", 
                                        gl.astype(ctype), 
                                        lt2_green.astype(ctype), 
                                        backend="jax")

        # t_iajb G_ir G_js Gp_pa Gp_qb L_pr L_qs type
        glgp = oe.contract("gir,rb->gib", 
                            gl.astype(ctype), 
                            greenp.astype(ctype), 
                            backend="jax")
    

        tauglgp = oe.contract("yia,gja->ygij", 
                              tau.astype(ctype),
                              glgp.astype(ctype), 
                              backend="jax")
        tr_tauglgp = oe.contract("ygii->yg", 
                              tauglgp.astype(ctype),
                              backend="jax")
        l2t2_c = oe.contract("yg,yg->", 
                             tr_tauglgp.astype(ctype), 
                             tr_tauglgp.astype(ctype), 
                             backend="jax").astype(jnp.complex128)
        l2t2_e = oe.contract("ygij,ygji->", 
                             tauglgp.astype(ctype), 
                             tauglgp.astype(ctype), 
                             backend="jax").astype(jnp.complex128)
        carry[3] += (2*l2t2_c - l2t2_e).astype(jnp.complex128)

        return carry, 0.0

    [e2_0, e2_2_2_1, e2_2_2_2, e2_2_3], _ = lax.scan(
        scan_chunk, [0.0, 0.0, 0.0, 0.0], (chol, rot_chol)
    )

    e2_2_1 = e2_0 * t2o
    e2_2_2 = 4 * (e2_2_2_1 + e2_2_2_2)
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3

    e0 = e1_0 + e2_0  # <psi|(h1+h2)|phi>/<psi|phi>
    e1 = e1_2 + e2_2  # <psi|t2(h1+h2)|phi>/<psi|phi>

    return obar, t2o, e0, e1

In [24]:
from afqmc import t2_tools
wave_data["tau"] = t2_tools.decompose_rt2(wave_data["t2"], 1e-6)
print(wave_data["tau"].shape)
print(f"rank reduction: "
      f"{wave_data['t2'].shape[0]*wave_data['t2'].shape[1]}"
      f" -> {wave_data['tau'].shape[0]}")

(30, 10, 6)
rank reduction: 60 -> 30


In [25]:
t2_rec = oe.contract('gia,gjb->iajb', wave_data["tau"], wave_data["tau"], backend='jax')
print(abs(t2_rec - wave_data["t2"]).max())

1.681209843340334e-16


In [57]:
walker = jnp.array(np.random.rand(*walker_init.shape))

obar, t2, e0, e1 = \
    trial._calc_energy_pt(walker, ham_data, wave_data)
print(pt2_energy_formula(h0, t2, e0, e1))

obar, t2, e0, e1 = \
    _calc_energy_pt2_decomposed(trial, walker, ham_data, wave_data)
print(pt2_energy_formula(h0, t2, e0, e1))

(-222.9099751010255+0j)
(-222.9099751010255+9.478497309794977e-16j)
